# Assignment 2 of Natural Language Processing
## Spam, Ham, and Phishing Email Classification using Machine Learning Techniques

Work assembled by Alejandro Gonçalves (202205564), Francisca Mihalache (202206022) and João Sousa (202205238).


## Table of Contents

1. [Introduction](#1-introduction)
   - 1.1. [Objectives and Scope](#11-objectives-and-scope)
   - 1.2. [Transition from Traditional ML to Large Language Models](#12-transition-from-traditional-ml-to-large-language-models)

2. [Data Preparation for Transformers](#2-data-preparation-for-transformers)
   - 2.1. [Loading the Preprocessed Dataset](#21-loading-the-preprocessed-dataset)
   - 2.2. [Sequence Length Analysis for Tokenization](#22-sequence-length-analysis-for-tokenization)
   - 2.3. [Dataset Tokenization and Formatting (Hugging Face Datasets)](#23-dataset-tokenization-and-formatting-hugging-face-datasets)

3. [Pre-trained Model Selection](#3-pre-trained-model-selection)
   - 3.1. [Justification for Model Choice (e.g., RoBERTa / SecBERT)](#31-justification-for-model-choice)
   - 3.2. [Loading the Base Architecture and Tokenizer](#32-loading-the-base-architecture-and-tokenizer)

4. [Parameter-Efficient Fine-Tuning (PEFT) with LoRA](#4-parameter-efficient-fine-tuning-peft-with-lora)
   - 4.1. [Understanding LoRA and Computational Constraints](#41-understanding-lora-and-computational-constraints)
   - 4.2. [Configuring the LoRA Adapters](#42-configuring-the-lora-adapters)

5. [Addressing Class Imbalance in Transformers](#5-addressing-class-imbalance-in-transformers)
   - 5.1. [Calculating Class Weights](#51-calculating-class-weights)
   - 5.2. [Implementing a Custom Trainer for Cost-Sensitive Learning](#52-implementing-a-custom-trainer-for-cost-sensitive-learning)

6. [Model Training and Evaluation](#6-model-training-and-evaluation)
   - 6.1. [Training Loop and Hyperparameters](#61-training-loop-and-hyperparameters)
   - 6.2. [Evaluation Metrics (Macro F1, Precision, Recall)](#62-evaluation-metrics-macro-f1-precision-recall)
   - 6.3. [Confusion Matrix Analysis](#63-confusion-matrix-analysis)

7. [Comparison: Traditional Models vs. Transformers](#7-comparison-traditional-models-vs-transformers)
   - 7.1. [Performance Comparison (Baseline vs. MLP vs. LoRA)](#71-performance-comparison-baseline-vs-mlp-vs-lora)
   - 7.2. [Computational Efficiency and Training Time](#72-computational-efficiency-and-training-time)

8. [Error Analysis: Did Contextual Embeddings Solve the Blind Spots?](#8-error-analysis-did-contextual-embeddings-solve-the-blind-spots)
   - 8.1. [Evaluating Bayesian Poisoning and Impersonation Cases](#81-evaluating-bayesian-poisoning-and-impersonation-cases)
   - 8.2. [Remaining Limitations](#82-remaining-limitations)

9. [Discussion/Conclusion](#9-discussionconclusion)

10. [References](#10-references)

### 1. Introduction
[[go back to the top]](#table-of-contents)

Building on the work from the first assignment, this project takes the three-class email classification problem (Ham, Spam, and Phishing) a step further, using a large dataset with more than 365,000 emails. Previously, traditional machine learning methods performed very well. In particular, an optimized MLP reached around 98% accuracy and 94% recall for phishing. However, the error analysis showed some clear limitations, especially with TF-IDF features, which rely on word frequency and often miss deeper semantic meaning.

In this second assignment, the approach changes significantly. Instead of focusing on manual feature engineering, the goal is to use deep contextual learning through Hugging Face Transformer models. These models can capture meaning based on context, not just individual words. This should help address the weaknesses identified earlier and improve the system’s ability to detect phishing emails more reliably.


#### 1.1. Objectives and Scope
[[go back to the topic]](#1-introduction)

The primary objective of this assignment is to design, fine-tune, and evaluate Transformer-based architectures for our specific cybersecurity classification task, strictly adhering to realistic computational constraints (a CPU-only training environment). The scope of this work includes:

* **Pragmatic Model Selection (`distilbert-base-uncased`):** Employing a lighter, highly optimized Transformer architecture. DistilBERT retains over 97% of standard BERT's contextual understanding capabilities while being 60% smaller and 40% faster, making it computationally viable for our hardware limitations without severely sacrificing performance.
* **Strategic Data Subsampling:** Implementing rigorous stratified sampling to train the Transformer on a highly representative subset of the overall 365,000-email corpus. This will test whether a context-aware model can generalize effectively and outperform traditional methods even when trained on a fraction of the original data volume.
* **Parameter-Efficient Fine-Tuning (PEFT):** Implementing Low-Rank Adaptation (LoRA) on the DistilBERT architecture to drastically reduce the number of trainable parameters, further optimizing the training pipeline for a CPU environment.
* **Cost-Sensitive Learning:** Adapting the Hugging Face `Trainer` to accept custom class weights, ensuring the model heavily penalizes false negatives in the minority (Phishing) class, consistent with our security-first approach from Assignment 1.
* **Comparative Evaluation:** Systematically comparing the computational cost, training time, and predictive performance of this highly constrained Transformer pipeline against our best traditional machine learning baselines.

#### 1.2. Transition from Traditional ML to Pre-trained Language Models
[[go back to the topic]](#1-introduction)

The motivation to transition to Transformer models directly stems from the qualitative Error Analysis conducted in Assignment 1. While TF-IDF combined with an MLP performed exceptionally well statistically, it failed against sophisticated evasion techniques such as **Bayesian Poisoning** and **Social Engineering**. 

Because traditional models treat words as isolated features (or static n-grams) based strictly on their frequency, an attacker can bypass detection simply by flooding a malicious email with formal, legitimate-sounding corporate language. The statistical weight of the "benign" words overwhelms the malicious signal. Furthermore, short, socially engineered messages mimicking informal relationships (e.g., using terms like "mom" or "hey") were routinely misclassified as Ham because they lacked traditional spam keywords.

Transformers resolve this by utilizing the self-attention mechanism. Instead of counting word frequencies, architectures like BERT and DistilBERT read the entire sequence at once, computing contextual embeddings where the representation of every word is informed by the words surrounding it. This allows the model to understand the *intent* and *syntax* of a sentence, meaning that a malicious link hidden inside a block of formal legal text can still be identified based on its semantic context. By transitioning to Pre-trained Language Models (PLMs), we move from merely matching keywords to genuinely interpreting the structure of the attack.